## Import Python Libraries

In [3]:
from pathlib import Path
import pandas as pd

## Set Project Folders

In [4]:
# Current notebook folder
current_directory = Path.cwd()

# Project root folder
project_root = current_directory.parent.parent

# eICU data folder
data_folder = project_root / "data"

print("Current working directory:")
print(current_directory)

print("\nProject root:")
print(project_root)

print("\nData folder:")
print(data_folder)

print("\nData folder exists:", data_folder.exists())

Current working directory:
C:\Users\samsa\Documents\ICU Clustering\notebooks\module_1

Project root:
C:\Users\samsa\Documents\ICU Clustering

Data folder:
C:\Users\samsa\Documents\ICU Clustering\data

Data folder exists: True


## Load and Inspect `patient.csv`

In [5]:
patient = pd.read_csv(data_folder / "patient.csv")
patient.head()

,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,wardid,apacheadmissiondx,admissionheight,hospitaladmittime24,...,unitadmitsource,unitvisitnumber,unitstaytype,admissionweight,dischargeweight,unitdischargetime24,unitdischargeoffset,unitdischargelocation,unitdischargestatus,uniquepid
0,141168,128919,Female,70,Caucasian,59,91,"Rhythm disturbance (atrial, supraventricular)",152.4,15:54:00,...,Direct Admit,1,admit,84.3,85.8,03:50:00,3596,Death,Expired,002-34851
1,141178,128927,Female,52,Caucasian,60,83,NaN,162.6,08:56:00,...,Emergency Department,1,admit,54.4,54.4,09:18:00,8,Step-Down Unit (SDU),Alive,002-33870
2,141179,128927,Female,52,Caucasian,60,83,NaN,162.6,08:56:00,...,ICU to SDU,2,stepdown/other,NaN,60.4,19:20:00,2042,Home,Alive,002-33870
3,141194,128941,Male,68,Caucasian,73,92,"Sepsis, renal/UTI (including bladder)",180.3,18:18:40,...,Floor,1,admit,73.9,76.7,15:31:00,4813,Floor,Alive,002-5276
4,141196,128943,Male,71,Caucasian,67,109,NaN,162.6,20:21:00,...,ICU to SDU,2,stepdown/other,NaN,63.2,22:23:00,1463,Floor,Alive,002-37665


#### Observation :

The `patient` table loaded correctly, and the first five rows show that it is the base ICU-stay table in eICU.

A few important things are already visible:
- `patientunitstayid` identifies a specific ICU stay.
- `patienthealthsystemstayid` identifies the broader hospital stay.
- The same hospital stay can contain more than one ICU/unit stay. For example, `patienthealthsystemstayid = 128927` appears with two different `patientunitstayid` values.
- The table contains demographics such as `gender`, `age`, and `ethnicity`.
- It also contains admission information such as `apacheadmissiondx`, `unitadmitsource`, and `unittype`.
- Outcome-related fields such as `unitdischargestatus`, `unitdischargelocation`, and `hospitaldischargestatus` are present.
- Some variables already show missing values, such as `apacheadmissiondx` and `admissionweight`.

This table will mainly be used for cohort definition, demographics, linking tables through `patientunitstayid`, and later outcome evaluation.

## Check Size and ICU-Stay uniqueness

In [6]:
print("Number of rows:", patient.shape[0])
print("Number of columns:", patient.shape[1])

print("\nUnique ICU stays:")
print(patient["patientunitstayid"].nunique())

print("\nDuplicate patientunitstayid values:")
print(patient["patientunitstayid"].duplicated().sum())

print("\nUnique hospital stays:")
print(patient["patienthealthsystemstayid"].nunique())

Number of rows: 200859
Number of columns: 29

Unique ICU stays:
200859

Duplicate patientunitstayid values:
0

Unique hospital stays:
166355


#### Observation :

The `patient` table contains 200,859 ICU stays and 29 columns.

The most important finding is:
- 200,859 rows
- 200,859 unique `patientunitstayid` values
- 0 duplicate `patientunitstayid` values

So, in this table, one row represents one ICU stay.

There are only 166,355 unique hospital stays, which is fewer than the number of ICU stays. This confirms that some patients have multiple ICU/unit stays during the same hospital admission.

For our project, this means `patientunitstayid` should be the main identifier when joining `patient.csv` with labs, vitals, medications, and other ICU tables.

## Check missingness in important patient variables

In [7]:
important_columns = [
    "gender",
    "age",
    "ethnicity",
    "apacheadmissiondx",
    "admissionheight",
    "admissionweight",
    "unittype",
    "unitadmitsource",
    "unitdischargestatus",
    "hospitaldischargestatus"
]

missing_values = patient[important_columns].isnull().sum()
missing_percent = patient[important_columns].isnull().mean() * 100

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing %": missing_percent.round(2)
})

missing_summary

,Missing Values,Missing %
gender,134,0.07
age,95,0.05
ethnicity,2290,1.14
apacheadmissiondx,22996,11.45
admissionheight,4215,2.10
admissionweight,16718,8.32
unittype,0,0.00
unitadmitsource,1090,0.54
unitdischargestatus,34,0.02
hospitaldischargestatus,1751,0.87


#### Observation :

The important patient-level variables are mostly well populated.
- `gender` and `age` have almost no missing data: $0.07\%$ and $0.05\%$.
- `ethnicity` is also fairly complete, with only $1.14\%$ missing.
- `unittype` is $100\%$ complete, which is useful if ICU type becomes part of cohort description or clustering.
- `unitadmitsource` has very little missingness at $0.54\%$.
- The main variables needing attention are:
    - `apacheadmissiondx` — $11.45\%$ missing
    - `admissionweight` — $8.32\%$ missing
    - `admissionheight` — $2.10\%$ missing


For the future clustering dataset, demographics such as age and gender should be relatively easy to use because coverage is very high. Admission weight may require a missing-data decision later.

The discharge-status variables are also highly complete, which is useful because we may later use mortality/outcome information to evaluate whether the discovered 12-hour and 24-hour clusters differ clinically.

## Inspect Important demographic and ICU categories

In [8]:
print("Age data type:")
print(patient["age"].dtype)

print("\nGender:")
print(patient["gender"].value_counts(dropna=False))

print("\nEthnicity:")
print(patient["ethnicity"].value_counts(dropna=False))

print("\nICU Unit Types:")
print(patient["unittype"].value_counts(dropna=False))

print("\nUnit Stay Types:")
print(patient["unitstaytype"].value_counts(dropna=False))

Age data type:
str

Gender:
gender
Male       108379
Female      92303
NaN           134
Unknown        35
Other           8
Name: count, dtype: int64

Ethnicity:
ethnicity
Caucasian           155285
African American     21308
Other/Unknown         9542
Hispanic              7464
Asian                 3270
NaN                   2290
Native American       1700
Name: count, dtype: int64

ICU Unit Types:
unittype
Med-Surg ICU    113222
MICU             17465
CCU-CTICU        15290
Neuro ICU        14451
Cardiac ICU      12467
SICU             12181
CSICU             9625
CTICU             6158
Name: count, dtype: int64

Unit Stay Types:
unitstaytype
admit             154948
stepdown/other     25239
transfer           10725
readmit             9947
Name: count, dtype: int64


#### Observation :
- `age` is stored as a text field, so it should be checked and converted carefully before analysis.
- Most ICU stays are initial admissions, while transfers and readmissions make up a smaller portion of the data.
- Med-Surg ICU is the most common unit type, with several specialized ICUs also represented.
- Gender and ethnicity are mostly complete, so these demographic variables are potentially usable.
- For future cohort construction, `unitstaytype` may help us decide whether to keep only initial ICU admissions or also include transfers/readmissions.

## Check Age Values

In [9]:
age_numeric = pd.to_numeric(patient["age"], errors="coerce")

print("Minimum numeric age:", age_numeric.min())
print("Maximum numeric age:", age_numeric.max())

print("\nNon-numeric or missing age values:")
print(patient.loc[age_numeric.isna(), "age"].value_counts(dropna=False))

Minimum numeric age: 0.0
Maximum numeric age: 89.0

Non-numeric or missing age values:
age
> 89    7081
NaN       95
Name: count, dtype: int64


#### Observation :
- Most age values are numeric and range from `0` to `89` years.
- There are $7081$ records coded as $\gt 89$ which explains why `age` is stored as text.
- There are only $95$ missing age values due to which `age` is highly complete overall.
- Before using age for clustering, we will need a consistent rule for handling $\gt 89$ and the small number of missing values.
- This is relevant for both the $12-$hour and $24-$hour analyses, since age will remain a fixed baseline variable in both datasets.

## Inspect discharge outcomes

In [10]:
print("Unit discharge status:")
print(patient["unitdischargestatus"].value_counts(dropna=False))

print("\nHospital discharge status:")
print(patient["hospitaldischargestatus"].value_counts(dropna=False))

Unit discharge status:
unitdischargestatus
Alive      189918
Expired     10907
NaN            34
Name: count, dtype: int64

Hospital discharge status:
hospitaldischargestatus
Alive      181104
Expired     18004
NaN          1751
Name: count, dtype: int64


#### Observation :
- Most ICU stays ended with the patient alive : $189,918$ while $10,917$ ICU stays ended in death.
- At the hospital level, $18,004$ stays ended in death which is higher because some patients survived the ICU but died later during the same hospitalization.
- Both outcome variables are highly complete especially `unitdischargestatus`.
- These variables should be used mainly for evaluating whether the $12-$ hour and $24-$ hour clusters differ in mortality rather than as inputs for creating the clusters. This gives us enough important information from `patient.csv`

## Load and inspect `apachePatientResult.csv`

In [11]:
apache_result = pd.read_csv(
    data_folder / "apachePatientResult.csv"
)

print("Shape:")
print(apache_result.shape)

print("\nColumns:")
print(apache_result.columns.tolist())

print("\nFirst 5 rows:")
apache_result.head()

Shape:
(297064, 23)

Columns:
['apachepatientresultsid', 'patientunitstayid', 'physicianspeciality', 'physicianinterventioncategory', 'acutephysiologyscore', 'apachescore', 'apacheversion', 'predictedicumortality', 'actualicumortality', 'predictediculos', 'actualiculos', 'predictedhospitalmortality', 'actualhospitalmortality', 'predictedhospitallos', 'actualhospitallos', 'preopmi', 'preopcardiaccath', 'ptcawithin24h', 'unabridgedunitlos', 'unabridgedhosplos', 'actualventdays', 'predventdays', 'unabridgedactualventdays']

First 5 rows:


,apachepatientresultsid,patientunitstayid,physicianspeciality,physicianinterventioncategory,acutephysiologyscore,apachescore,apacheversion,predictedicumortality,actualicumortality,predictediculos,...,predictedhospitallos,actualhospitallos,preopmi,preopcardiaccath,ptcawithin24h,unabridgedunitlos,unabridgedhosplos,actualventdays,predventdays,unabridgedactualventdays
0,26570,141168,critical care medicine (CCM),Unknown,49,65,IV,0.026988,EXPIRED,3.038388,...,7.546453,2.4972,0,0,0,2.4972,2.4972,NaN,NaN,NaN
1,26571,141168,critical care medicine (CCM),Unknown,49,65,IVa,0.028889,EXPIRED,3.091127,...,6.628720,2.4972,0,0,0,2.4972,2.4972,NaN,NaN,NaN
2,53135,141194,critical care medicine (CCM),Unknown,57,70,IV,0.037888,ALIVE,4.620982,...,13.338449,9.2167,0,0,0,3.3423,9.2167,NaN,NaN,NaN
3,53136,141194,critical care medicine (CCM),Unknown,57,70,IVa,0.046448,ALIVE,4.167129,...,12.978228,9.2167,0,0,0,3.3423,9.2167,NaN,NaN,NaN
4,8,141203,hospitalist,I,73,90,IVa,0.291609,ALIVE,8.670299,...,16.319389,3.7493,0,0,0,1.2979,3.7493,2.0,5.738093,2.0


#### Observation :
- The table contains $297,064$ rows and $23$ columns, so it has more rows than `patient.csv`
- The sample shows that the same `patientunitstayid` can appear more than once because APACHE results may be available for different versions such as `IV` and `IVa`.
- Important fields include `apachescore`, `acutephysiologyscore`, predicted and actual ICU mortality, predicted and actual hospital mortality and ICU/hospital length of stay.
- This table will be most useful for severity assessment and evaluating whether the $12-$ hour and $24-$ hour clusters differ in outcomes, rather than as a direct source of time-windowed measurements.

## Check ICU-stay coverage and APACHE versions

In [12]:
print("Unique ICU stays:")
print(apache_result["patientunitstayid"].nunique())

print("\nDuplicate patientunitstayid values:")
print(apache_result["patientunitstayid"].duplicated().sum())

print("\nAPACHE versions:")
print(apache_result["apacheversion"].value_counts(dropna=False))

Unique ICU stays:
148532

Duplicate patientunitstayid values:
148532

APACHE versions:
apacheversion
IV     148532
IVa    148532
Name: count, dtype: int64


#### Observations :
- The table covers $148532$ unique ICU stays which is fewer than the $200859$ stays in `patient.csv`
- Every included ICU stay appears twice : once for APACHE `IV` and once for APACHE `IVa`
- So this table is not one-row-per-ICU stay unless we first choose which APACHE version to use.
- This matters later when joining with other tables, because keeping both versions would duplicate patients.
- For cluster evaluation, we will likely use one consistent APACHE version to compare severity, mortality and length of stay across the $12-hour$ and $24-hour$ clusters.

## Check missingness in important APACHE variables

In [13]:
important_apache_columns = [
    "acutephysiologyscore",
    "apachescore",
    "predictedicumortality",
    "actualicumortality",
    "predictediculos",
    "actualiculos",
    "predictedhospitalmortality",
    "actualhospitalmortality",
    "predictedhospitallos",
    "actualhospitallos"
]

missing_values = apache_result[important_apache_columns].isnull().sum()
missing_percent = apache_result[important_apache_columns].isnull().mean() * 100

apache_missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing %": missing_percent.round(2)
})

apache_missing_summary

,Missing Values,Missing %
acutephysiologyscore,0,0.0
apachescore,0,0.0
predictedicumortality,0,0.0
actualicumortality,0,0.0
predictediculos,0,0.0
actualiculos,0,0.0
predictedhospitalmortality,0,0.0
actualhospitalmortality,0,0.0
predictedhospitallos,0,0.0
actualhospitallos,0,0.0


#### Observation :
- All major APACHE severity, mortality, and length-of-stay variables have $0\%$ missingness within this table.
- However, the table covers only $148532$ ICU stays, so APACHE results are not available for every stay in `patient.csv`
- These variables are therefore strong candidates for cluster validation and severity comparison but using them could reduce the available cohort if APACHE data are required.
- Because each stay has both `IV` and `IVa` records, we should use one consistent APACHE version when analyzing outcomes.

## Inspect key APACHE values using IVa only

In [14]:
apache_iva = apache_result[
    apache_result["apacheversion"] == "IVa"
]

key_apache_variables = [
    "acutephysiologyscore",
    "apachescore",
    "predictedhospitalmortality",
    "actualiculos",
    "actualhospitallos"
]

print("Key APACHE IVa summary:")
print(apache_iva[key_apache_variables].describe().round(2))

print("\nActual hospital mortality:")
print(apache_iva["actualhospitalmortality"].value_counts())

Key APACHE IVa summary:
       acutephysiologyscore  apachescore  predictedhospitalmortality  \
count             148532.00    148532.00                   148532.00   
mean                  43.07        54.80                        0.03   
std                   24.00        26.13                        0.35   
min                   -1.00        -1.00                       -1.00   
25%                   27.00        37.00                        0.02   
50%                   38.00        51.00                        0.05   
75%                   54.00        68.00                        0.12   
max                  200.00       211.00                        1.00   

       actualiculos  actualhospitallos  
count     148532.00          148532.00  
mean           3.00               8.05  
std            4.30               8.15  
min            0.17               0.11  
25%            0.99               2.94  
50%            1.81               5.51  
75%            3.29               9.98  

#### Observation :
- APACHE IVa includes $148532$ ICU stays, with a median APACHE score of $51$ and median ICU LOS of $1.81$ days.
- Hospital mortality in this APACHE subset is $13737$ deaths versus $134795$ survivors
- Although pandas reported $0\%$ missingness, several variables have a minimum value of $-1$.
- This suggests $-1$ may represent an unavailable or special-coded value rather than a true clinical measurement, so we should quantify it before using these variables.
- APACHE severity and outcomes remain most appropriate for cluster validation, not as core clustering features.

## Check `-1` values in APACHE variables


In [15]:
check_columns = [
    "acutephysiologyscore",
    "apachescore",
    "predictedicumortality",
    "predictedhospitalmortality",
    "predictediculos",
    "predictedhospitallos"
]

for column in check_columns:
    count = (apache_iva[column] == -1).sum()
    percent = (count / len(apache_iva)) * 100
    
    print(column, ":", count, f"({percent:.2f}%)")

acutephysiologyscore : 1836 (1.24%)
apachescore : 1836 (1.24%)
predictedicumortality : 4487 (3.02%)
predictedhospitalmortality : 12296 (8.28%)
predictediculos : 4487 (3.02%)
predictedhospitallos : 12296 (8.28%)


#### Observation :
- The $-1$ values confirm that standard null checking alone is not enough for APACHE variables.
- About $1.24\%$ of APACHE/physiology scores and $3–8\%$ of prediction fields contain $-1$.
- These values should not be treated as real clinical measurements. We should verify their exact definition in the eICU data dictionary before preprocessing.
- Overall, this table is valuable for severity and outcome validation, but requires handling of special-coded values first.

## Load and inspect `apachePredVar.csv`

In [16]:
apache_pred = pd.read_csv(
    data_folder / "apachePredVar.csv"
)

print("Shape:")
print(apache_pred.shape)

print("\nColumns:")
print(apache_pred.columns.tolist())

print("\nFirst 5 rows:")
apache_pred.head()

Shape:
(171177, 51)

Columns:
['apachepredvarid', 'patientunitstayid', 'sicuday', 'saps3day1', 'saps3today', 'saps3yesterday', 'gender', 'teachtype', 'region', 'bedcount', 'admitsource', 'graftcount', 'meds', 'verbal', 'motor', 'eyes', 'age', 'admitdiagnosis', 'thrombolytics', 'diedinhospital', 'aids', 'hepaticfailure', 'lymphoma', 'metastaticcancer', 'leukemia', 'immunosuppression', 'cirrhosis', 'electivesurgery', 'activetx', 'readmit', 'ima', 'midur', 'ventday1', 'oobventday1', 'oobintubday1', 'diabetes', 'managementsystem', 'var03hspxlos', 'pao2', 'fio2', 'ejectfx', 'creatinine', 'dischargelocation', 'visitnumber', 'amilocation', 'day1meds', 'day1verbal', 'day1motor', 'day1eyes', 'day1pao2', 'day1fio2']

First 5 rows:


,apachepredvarid,patientunitstayid,sicuday,saps3day1,saps3today,saps3yesterday,gender,teachtype,region,bedcount,...,creatinine,dischargelocation,visitnumber,amilocation,day1meds,day1verbal,day1motor,day1eyes,day1pao2,day1fio2
0,1794895,141168,1,0,0,0,1,0,3,12,...,2.30,9,1,-1,0,5,6,4,-1.0,-1.0
1,2406430,141178,1,0,0,0,1,0,3,9,...,-1.00,4,1,-1,-1,-1,-1,-1,-1.0,-1.0
2,1790923,141194,1,0,0,0,0,0,3,38,...,2.51,4,1,-1,0,4,6,3,-1.0,-1.0
3,27799,141197,1,0,0,0,0,0,3,30,...,-1.00,4,1,-1,0,5,6,4,-1.0,-1.0
4,2406432,141203,1,0,0,0,1,0,3,18,...,0.56,4,1,-1,0,1,3,1,51.0,100.0


#### Observation :
- The table contains $171177$ rows and $51$ columns with demographic, severity, comorbidity, ventilation, neurologic, and selected physiologic variables.
- Several fields already show $-1$ values, so special-coded missing or unavailable values will need attention.
- Variables such as `ventday1`, `day1pao2`, and `day1fio2` appear to represent day-1 summaries, so they are not suitable for a clean 12-hour versus 24-hour comparison without confirming their exact definitions.
- This table may be useful for baseline severity, comorbidities, and validation, but the main 12h/24h clustering features will likely come more directly from vitals and labs.

## Check ICU-stay uniqueness

In [17]:
print("Unique ICU stays:")
print(apache_pred["patientunitstayid"].nunique())

print("\nDuplicate patientunitstayid values:")
print(apache_pred["patientunitstayid"].duplicated().sum())

print("\nSICU day values:")
print(apache_pred["sicuday"].value_counts(dropna=False).sort_index())

Unique ICU stays:
171177

Duplicate patientunitstayid values:
0

SICU day values:
sicuday
1    171177
Name: count, dtype: int64


#### Observation :
- The table has $171,177$ unique ICU stays, with no duplicate `patientunitstayid` values, so it is one row per ICU stay.
- Every record has `sicuday = 1`, confirming this table contains day-1 APACHE prediction variables.
- Because these are summarized day-1 fields rather than timestamped measurements, we should not assume they represent exactly 12 or 24 hours without checking the data dictionary.
- The table can still provide useful severity, ventilation, and comorbidity information for cohort description or later validation.

## Check missing and `-1` values in important variables


In [18]:
important_pred_columns = [
    "age",
    "creatinine",
    "pao2",
    "fio2",
    "ventday1",
    "day1pao2",
    "day1fio2",
    "diabetes",
    "cirrhosis",
    "hepaticfailure",
    "immunosuppression"
]

for column in important_pred_columns:
    missing = apache_pred[column].isnull().sum()
    minus_one = (apache_pred[column] == -1).sum()

    print(
        column,
        "- Missing:", missing,
        "| -1 values:", minus_one,
        f"({minus_one / len(apache_pred) * 100:.2f}%)"
    )

age - Missing: 6084 | -1 values: 0 (0.00%)
creatinine - Missing: 0 | -1 values: 37090 (21.67%)
pao2 - Missing: 0 | -1 values: 132105 (77.17%)
fio2 - Missing: 0 | -1 values: 132105 (77.17%)
ventday1 - Missing: 0 | -1 values: 0 (0.00%)
day1pao2 - Missing: 0 | -1 values: 132105 (77.17%)
day1fio2 - Missing: 0 | -1 values: 132105 (77.17%)
diabetes - Missing: 0 | -1 values: 0 (0.00%)
cirrhosis - Missing: 0 | -1 values: 0 (0.00%)
hepaticfailure - Missing: 0 | -1 values: 0 (0.00%)
immunosuppression - Missing: 0 | -1 values: 0 (0.00%)


#### Observation :
- `age` has relatively little missingness: $6,084 records (~3.6%)$.
- `creatinine` has $21.67\%$ coded as $-1$, so its usable coverage is limited.
- `pao2` and `fio2` are unavailable ($-1$) for about $77\%$ of stays, making them poor primary clustering variables from this table.
- Comorbidity indicators such as diabetes, cirrhosis, hepatic failure, and immunosuppression are complete.
- Overall, apachePredVar is more useful for baseline/severity information than for constructing the main time-windowed physiological features.

## Inspect `vitalPeriodic.csv` structure


In [19]:
vital_periodic_sample = pd.read_csv(
    data_folder / "vitalPeriodic.csv",
    nrows=5
)

print("Columns:")
print(vital_periodic_sample.columns.tolist())

print("\nFirst 5 rows:")
vital_periodic_sample

Columns:
['vitalperiodicid', 'patientunitstayid', 'observationoffset', 'temperature', 'sao2', 'heartrate', 'respiration', 'cvp', 'etco2', 'systemicsystolic', 'systemicdiastolic', 'systemicmean', 'pasystolic', 'padiastolic', 'pamean', 'st1', 'st2', 'st3', 'icp']

First 5 rows:


,vitalperiodicid,patientunitstayid,observationoffset,temperature,sao2,heartrate,respiration,cvp,etco2,systemicsystolic,systemicdiastolic,systemicmean,pasystolic,padiastolic,pamean,st1,st2,st3,icp
0,37376747,141168,2059,NaN,NaN,92,NaN,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,37404957,141168,1289,NaN,NaN,118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,37385871,141168,1794,NaN,91.0,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,37401664,141168,1374,NaN,90.0,118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,37377404,141168,2039,NaN,98.0,92,NaN,33.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Observation :
- This is a repeated-measurement table: the same ICU stay appears many times at different `observationoffset` values.
- It contains important continuous vital signs such as `heart rate`, `respiratory rate`, `temperature`, `SpO₂`, and several invasive hemodynamic measurements.
- Many fields are missing in individual rows because not every vital is recorded at every timestamp.
- The sample includes measurements occurring beyond the first 24 hours, so we must explicitly filter by `observationoffset`.
- This table will be one of the main sources for constructing comparable 12-hour and 24-hour physiological features.


## Check overall and 12h/24h coverage

In [20]:
vital_file = data_folder / "vitalPeriodic.csv"

total_rows = 0

all_stays = set()
stays_12h = set()
stays_24h = set()

min_offset = None
max_offset = None

for chunk in pd.read_csv(
    vital_file,
    usecols=["patientunitstayid", "observationoffset"],
    chunksize=1_000_000
):
    total_rows += len(chunk)

    all_stays.update(chunk["patientunitstayid"].unique())

    data_12h = chunk[
        (chunk["observationoffset"] >= 0) &
        (chunk["observationoffset"] <= 720)
    ]

    data_24h = chunk[
        (chunk["observationoffset"] >= 0) &
        (chunk["observationoffset"] <= 1440)
    ]

    stays_12h.update(data_12h["patientunitstayid"].unique())
    stays_24h.update(data_24h["patientunitstayid"].unique())

    chunk_min = chunk["observationoffset"].min()
    chunk_max = chunk["observationoffset"].max()

    min_offset = chunk_min if min_offset is None else min(min_offset, chunk_min)
    max_offset = chunk_max if max_offset is None else max(max_offset, chunk_max)

print("Total rows:", total_rows)
print("Unique ICU stays:", len(all_stays))

print("\nObservation offset range:")
print("Minimum:", min_offset)
print("Maximum:", max_offset)

print("\nICU stays with data in first 12 hours:", len(stays_12h))
print("ICU stays with data in first 24 hours:", len(stays_24h))

Total rows: 146671642
Unique ICU stays: 192831

Observation offset range:
Minimum: -49781
Maximum: 766055

ICU stays with data in first 12 hours: 191364
ICU stays with data in first 24 hours: 192240


#### Observation :
- This is a very large table with about $146.7$ million measurements across $192,831$ ICU stays.
- `observationoffset` ranges from negative values to far beyond 24 hours, so explicit time-window filtering is essential.
- $191364$ stays $(99.2\%)$ have at least one periodic-vital record in the first 12 hours.
- $192,240$ stays $(99.7\%)$ have at least one record in the first 24 hours. Overall coverage is excellent, but this does not mean every vital sign is available for every patient.
- Negative offsets represent measurements before the ICU admission reference point and should not be included in our first-12h/24h feature windows.

## Check availability of important vital signs

In [21]:
vital_columns = [
    "temperature",
    "sao2",
    "heartrate",
    "respiration"
]

available_12h = {column: set() for column in vital_columns}
available_24h = {column: set() for column in vital_columns}

for chunk in pd.read_csv(
    vital_file,
    usecols=["patientunitstayid", "observationoffset"] + vital_columns,
    chunksize=1_000_000
):
    data_12h = chunk[
        (chunk["observationoffset"] >= 0) &
        (chunk["observationoffset"] <= 720)
    ]

    data_24h = chunk[
        (chunk["observationoffset"] >= 0) &
        (chunk["observationoffset"] <= 1440)
    ]

    for column in vital_columns:
        stays = data_12h.loc[
            data_12h[column].notna(),
            "patientunitstayid"
        ]
        available_12h[column].update(stays.unique())

        stays = data_24h.loc[
            data_24h[column].notna(),
            "patientunitstayid"
        ]
        available_24h[column].update(stays.unique())

print("ICU stays with at least one measurement:\n")

for column in vital_columns:
    print(
        column,
        "| 12h:", len(available_12h[column]),
        "| 24h:", len(available_24h[column])
    )

ICU stays with at least one measurement:

temperature | 12h: 15717 | 24h: 16852
sao2 | 12h: 187571 | 24h: 188796
heartrate | 12h: 190801 | 24h: 191673
respiration | 12h: 175008 | 24h: 176582


#### Observation :
- `Heart rate` has excellent coverage in both windows: about $99.7\%$ of ICU stays.
- `SpO₂` is also highly available: about $98\%$.
- Respiratory rate has good coverage at roughly $91–92\%$.
- Temperature is very sparse in this table, available for only about $8–9\%$ of stays, so vitalPeriodic alone may not be a reliable temperature source.
- Coverage increases only slightly from $12h$ to $24h$, meaning the main periodic vitals are already captured for most patients within the first $12$ hours.

Overall, `heart rate`, `SpO₂`, and `respiratory rate` are strong candidate variables from this table for both clustering windows.

## Inspect `vitalAperiodic.csv`

In [22]:
vital_aperiodic_sample = pd.read_csv(
    data_folder / "vitalAperiodic.csv",
    nrows=5
)

print("Columns:")
print(vital_aperiodic_sample.columns.tolist())

print("\nFirst 5 rows:")
vital_aperiodic_sample

Columns:
['vitalaperiodicid', 'patientunitstayid', 'observationoffset', 'noninvasivesystolic', 'noninvasivediastolic', 'noninvasivemean', 'paop', 'cardiacoutput', 'cardiacinput', 'svr', 'svri', 'pvr', 'pvri']

First 5 rows:


,vitalaperiodicid,patientunitstayid,observationoffset,noninvasivesystolic,noninvasivediastolic,noninvasivemean,paop,cardiacoutput,cardiacinput,svr,svri,pvr,pvri
0,4295739,141168,349,NaN,NaN,79,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4295737,141168,123,106.0,68.0,81,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4295741,141168,1398,NaN,NaN,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4295740,141168,441,NaN,NaN,62,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4295738,141168,138,111.0,62.0,82,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Observation :
- This is another repeated-measurement table, with measurements linked to ICU stays through `patientunitstayid` and timed by `observationoffset`.
- Its main useful variables are non-invasive systolic, diastolic, and mean blood pressure.
- More specialized measures such as cardiac output and vascular resistance appear much sparser and may not be useful for the main clustering cohort.
- Like `vitalPeriodic`, this table must be filtered separately for the first 12 hours and first 24 hours.

## Check 12h/24h blood-pressure coverage

In [23]:
vital_aperiodic_file = data_folder / "vitalAperiodic.csv"

bp_columns = [
    "noninvasivesystolic",
    "noninvasivediastolic",
    "noninvasivemean"
]

total_rows = 0
all_stays = set()

available_12h = {column: set() for column in bp_columns}
available_24h = {column: set() for column in bp_columns}

for chunk in pd.read_csv(
    vital_aperiodic_file,
    usecols=["patientunitstayid", "observationoffset"] + bp_columns,
    chunksize=1_000_000
):
    total_rows += len(chunk)
    all_stays.update(chunk["patientunitstayid"].unique())

    data_12h = chunk[
        (chunk["observationoffset"] >= 0) &
        (chunk["observationoffset"] <= 720)
    ]

    data_24h = chunk[
        (chunk["observationoffset"] >= 0) &
        (chunk["observationoffset"] <= 1440)
    ]

    for column in bp_columns:
        available_12h[column].update(
            data_12h.loc[data_12h[column].notna(), "patientunitstayid"].unique()
        )

        available_24h[column].update(
            data_24h.loc[data_24h[column].notna(), "patientunitstayid"].unique()
        )

print("Total rows:", total_rows)
print("Unique ICU stays:", len(all_stays))

print("\nICU stays with at least one measurement:")
for column in bp_columns:
    print(
        column,
        "| 12h:", len(available_12h[column]),
        "| 24h:", len(available_24h[column])
    )

Total rows: 25075074
Unique ICU stays: 189753

ICU stays with at least one measurement:
noninvasivesystolic | 12h: 183159 | 24h: 186737
noninvasivediastolic | 12h: 183163 | 24h: 186741
noninvasivemean | 12h: 183213 | 24h: 186785


#### Observation :
- This table is also very large, with $25.1$ million rows covering $189,753$ ICU stays.
- Non-invasive systolic, diastolic, and mean blood pressure all have excellent coverage.
- Around $183k$ stays have BP measurements within 12h, increasing to about $187k$ within 24h.
- The small increase from 12h to 24h suggests blood pressure is captured early for most patients.
- These three BP variables are therefore strong candidates for both the $12-hour$ and $24-hour$ clustering datasets.

## Load and Inspect `lab.csv`

In [24]:
lab_sample = pd.read_csv(
    data_folder / "lab.csv",
    nrows=5
)

print("Columns:")
print(lab_sample.columns.tolist())

print("\nFirst 5 rows:")
lab_sample

Columns:
['labid', 'patientunitstayid', 'labresultoffset', 'labtypeid', 'labname', 'labresult', 'labresulttext', 'labmeasurenamesystem', 'labmeasurenameinterface', 'labresultrevisedoffset']

First 5 rows:


,labid,patientunitstayid,labresultoffset,labtypeid,labname,labresult,labresulttext,labmeasurenamesystem,labmeasurenameinterface,labresultrevisedoffset
0,52307161,141168,2026,3,fibrinogen,177.0,177.0,mg/dL,mg/dL,2219
1,50363251,141168,1133,3,PT - INR,2.5,2.5,ratio,NaN,1208
2,49149139,141168,2026,1,magnesium,2.0,2.0,mg/dL,mg/dL,2090
3,50363250,141168,1133,3,PT,26.6,26.6,sec,sec,1208
4,66695374,141168,2141,7,pH,7.2,7.2,NaN,Units,2155


#### Observation :
- This is a repeated laboratory-measurement table, linked by `patientunitstayid`.
- `labresultoffset` gives the timing of each lab result, so we can create separate 12-hour and 24-hour lab features.
- `labname` identifies the test, while `labresult` contains the numeric value and the measurement-name fields provide units.
- A patient can have the same lab measured multiple times, so later we can summarize values using statistics such as minimum, maximum, mean, or most abnormal value.
- `labresultrevisedoffset` is separate from the original result time; for our window definition, `labresultoffset` is the key timing field.

## Check overall and 12h/24h lab coverage

In [25]:
lab_file = data_folder / "lab.csv"

total_rows = 0
all_stays = set()
stays_12h = set()
stays_24h = set()

for chunk in pd.read_csv(
    lab_file,
    usecols=["patientunitstayid", "labresultoffset"],
    chunksize=1_000_000
):
    total_rows += len(chunk)

    all_stays.update(chunk["patientunitstayid"].unique())

    data_12h = chunk[
        (chunk["labresultoffset"] >= 0) &
        (chunk["labresultoffset"] <= 720)
    ]

    data_24h = chunk[
        (chunk["labresultoffset"] >= 0) &
        (chunk["labresultoffset"] <= 1440)
    ]

    stays_12h.update(data_12h["patientunitstayid"].unique())
    stays_24h.update(data_24h["patientunitstayid"].unique())

print("Total rows:", total_rows)
print("Unique ICU stays:", len(all_stays))

print("\nICU stays with lab data in first 12 hours:", len(stays_12h))
print("ICU stays with lab data in first 24 hours:", len(stays_24h))

Total rows: 39132531
Unique ICU stays: 195730

ICU stays with lab data in first 12 hours: 177999
ICU stays with lab data in first 24 hours: 189308


#### Observation :
- `lab.csv` is a large table with $39.1$ million rows covering $195,730$ ICU stays.
- About $178,000$ stays have at least one lab result within 12 hours, increasing to $189,000$ within 24 hours.
- Relative to the full patient table, this is roughly $89\%$ coverage at 12h and $94\%$ at 24h.
- Lab coverage improves more noticeably from 12h to 24h than vital-sign coverage, so the 24-hour dataset may retain more patients.
- Because overall lab coverage does not tell us which specific tests are available, the next important step is to inspect the most common lab names in each window.

## Find the most common labs in 12h and 24h

In [26]:
lab_counts_12h = {}
lab_counts_24h = {}

for chunk in pd.read_csv(
    lab_file,
    usecols=["labresultoffset", "labname"],
    chunksize=1_000_000
):
    data_12h = chunk[
        (chunk["labresultoffset"] >= 0) &
        (chunk["labresultoffset"] <= 720)
    ]

    data_24h = chunk[
        (chunk["labresultoffset"] >= 0) &
        (chunk["labresultoffset"] <= 1440)
    ]

    counts_12h = data_12h["labname"].value_counts()
    counts_24h = data_24h["labname"].value_counts()

    for lab_name, count in counts_12h.items():
        lab_counts_12h[lab_name] = lab_counts_12h.get(lab_name, 0) + count

    for lab_name, count in counts_24h.items():
        lab_counts_24h[lab_name] = lab_counts_24h.get(lab_name, 0) + count

print("Top 15 labs in first 12 hours:")
print(pd.Series(lab_counts_12h).sort_values(ascending=False).head(15))

print("\nTop 15 labs in first 24 hours:")
print(pd.Series(lab_counts_24h).sort_values(ascending=False).head(15))

Top 15 labs in first 12 hours:
bedside glucose     348174
potassium           196490
sodium              181265
glucose             170153
Hgb                 167705
Hct                 164187
chloride            163422
creatinine          161776
BUN                 160547
calcium             155456
bicarbonate         152558
platelets x 1000    139784
WBC x 1000          137890
RBC                 136893
MCV                 133806
dtype: int64

Top 15 labs in first 24 hours:
bedside glucose     642808
potassium           320742
sodium              295398
glucose             278074
Hgb                 274829
Hct                 269011
chloride            268850
creatinine          266260
BUN                 264423
calcium             257188
bicarbonate         251983
platelets x 1000    229054
WBC x 1000          225912
RBC                 224788
MCV                 219635
dtype: int64


#### Observation :
- The same core laboratory tests dominate both 12-hour and 24-hour windows, which is good for making the two clustering analyses comparable.
- Common tests include sodium, potassium, glucose, hemoglobin, chloride, creatinine, BUN, bicarbonate, platelets, and WBC.
- The 24-hour window naturally contains many more measurements because patients have more time for repeat testing.
- These counts represent number of lab measurements, not number of patients. Before selecting features, we need to check how many ICU stays actually have each important lab available.

## Check patient-level coverage of important labs

In [27]:
important_labs = [
    "sodium",
    "potassium",
    "glucose",
    "Hgb",
    "chloride",
    "creatinine",
    "BUN",
    "bicarbonate",
    "platelets x 1000",
    "WBC x 1000",
    "lactate",
    "anion gap"
]

lab_stays_12h = {lab: set() for lab in important_labs}
lab_stays_24h = {lab: set() for lab in important_labs}

for chunk in pd.read_csv(
    lab_file,
    usecols=["patientunitstayid", "labresultoffset", "labname"],
    chunksize=1_000_000
):
    data_12h = chunk[
        (chunk["labresultoffset"] >= 0) &
        (chunk["labresultoffset"] <= 720)
    ]

    data_24h = chunk[
        (chunk["labresultoffset"] >= 0) &
        (chunk["labresultoffset"] <= 1440)
    ]

    for lab in important_labs:
        stays_12h = data_12h.loc[
            data_12h["labname"] == lab,
            "patientunitstayid"
        ]

        stays_24h = data_24h.loc[
            data_24h["labname"] == lab,
            "patientunitstayid"
        ]

        lab_stays_12h[lab].update(stays_12h.unique())
        lab_stays_24h[lab].update(stays_24h.unique())

coverage = pd.DataFrame({
    "12h stays": [len(lab_stays_12h[lab]) for lab in important_labs],
    "24h stays": [len(lab_stays_24h[lab]) for lab in important_labs]
}, index=important_labs)

coverage

,12h stays,24h stays
sodium,122855,170012
potassium,127637,171455
glucose,120718,168848
Hgb,117903,164586
chloride,119999,168867
creatinine,120781,169966
BUN,120257,169304
bicarbonate,113294,159966
platelets x 1000,111747,162374
WBC x 1000,111586,162721


#### Observation :
- Core labs such as sodium, potassium, glucose, creatinine, BUN, hemoglobin, WBC, and platelets have substantially better coverage at 24h than at 12h.
- At 12h, most common labs cover roughly $55–64\%$ of all ICU stays; by 24h many reach about $80–85\%$.
- `anion gap` has moderate coverage, while `lactate` is much sparser: only $38,976$ stays at 12h and $44,034$ at 24h.
- Therefore, the common chemistry/CBC labs are strong clustering candidates. Lactate is clinically useful but requiring it could exclude many patients.
- The larger increase from 12h to 24h means lab missingness will be an important difference between the two clustering windows.

## Load and inspect `diagnosis.csv`

In [28]:
diagnosis = pd.read_csv(
    data_folder / "diagnosis.csv"
)

print("Shape:")
print(diagnosis.shape)

print("\nColumns:")
print(diagnosis.columns.tolist())

print("\nFirst 5 rows:")
diagnosis.head()

Shape:
(2710672, 7)

Columns:
['diagnosisid', 'patientunitstayid', 'activeupondischarge', 'diagnosisoffset', 'diagnosisstring', 'icd9code', 'diagnosispriority']

First 5 rows:


,diagnosisid,patientunitstayid,activeupondischarge,diagnosisoffset,diagnosisstring,icd9code,diagnosispriority
0,4222318,141168,False,72,cardiovascular|chest pain / ASHD|coronary arte...,"414.00, I25.10",Other
1,3370568,141168,True,118,cardiovascular|ventricular disorders|cardiomyo...,NaN,Other
2,4160941,141168,False,72,pulmonary|disorders of the airways|COPD,"491.20, J44.9",Other
3,4103261,141168,True,118,pulmonary|disorders of the airways|COPD,"491.20, J44.9",Other
4,3545241,141168,True,118,cardiovascular|ventricular disorders|congestiv...,"428.0, I50.9",Other


#### Observation :
- This is a repeated diagnosis table with $2.71$ million rows, so each ICU stay can have multiple diagnoses.
- `diagnosisoffset` records when the diagnosis was documented, which lets us compare diagnoses available within the first 12h and 24h.
- `diagnosisstring` contains hierarchical clinical descriptions, while `icd9code` provides coded diagnoses.
- Diagnoses can help with clinical interpretation of clusters, but they should be used cautiously as clustering inputs because documentation timing can vary.
- `activeupondischarge` and `diagnosispriority` may also help distinguish active conditions and diagnosis importance.

## Check diagnosis coverage in 12h and 24h

In [29]:
print("Unique ICU stays:")
print(diagnosis["patientunitstayid"].nunique())

diagnosis_12h = diagnosis[
    (diagnosis["diagnosisoffset"] >= 0) &
    (diagnosis["diagnosisoffset"] <= 720)
]

diagnosis_24h = diagnosis[
    (diagnosis["diagnosisoffset"] >= 0) &
    (diagnosis["diagnosisoffset"] <= 1440)
]

print("\nICU stays with diagnosis in first 12 hours:")
print(diagnosis_12h["patientunitstayid"].nunique())

print("\nICU stays with diagnosis in first 24 hours:")
print(diagnosis_24h["patientunitstayid"].nunique())

print("\nDiagnosis priority:")
print(diagnosis["diagnosispriority"].value_counts(dropna=False))

Unique ICU stays:
173109

ICU stays with diagnosis in first 12 hours:
167822

ICU stays with diagnosis in first 24 hours:
170813

Diagnosis priority:
diagnosispriority
Other      1140846
Major      1119718
Primary     450108
Name: count, dtype: int64


#### Observation :
- The table covers $173,109$ ICU stays, and most diagnoses are documented early.
- $167,822$ stays have at least one diagnosis within 12h, increasing only slightly to $170,813$ within 24h.
- Diagnosis coverage is therefore already strong in the first 12 hours.
- The table contains a mix of Primary, Major, and Other diagnoses, with Major and Other diagnoses being most frequent.
- For this project, diagnoses will be especially useful for describing and clinically interpreting the clusters, rather than as core physiological clustering features.

## Check common primary diagnoses in 12h and 24h

In [30]:
primary_12h = diagnosis_12h[
    diagnosis_12h["diagnosispriority"] == "Primary"
]

primary_24h = diagnosis_24h[
    diagnosis_24h["diagnosispriority"] == "Primary"
]

print("Top 10 primary diagnoses in first 12 hours:")
print(primary_12h["diagnosisstring"].value_counts().head(10))

print("\nTop 10 primary diagnoses in first 24 hours:")
print(primary_24h["diagnosisstring"].value_counts().head(10))

Top 10 primary diagnoses in first 12 hours:
diagnosisstring
cardiovascular|shock / hypotension|sepsis                                                                 8084
pulmonary|respiratory failure|acute respiratory failure                                                   7656
cardiovascular|ventricular disorders|congestive heart failure                                             4356
endocrine|glucose metabolism|DKA                                                                          4328
cardiovascular|shock / hypotension|sepsis|severe                                                          3932
pulmonary|pulmonary infections|pneumonia                                                                  3613
cardiovascular|cardiac surgery|s/p CABG < 7 days                                                          3247
cardiovascular|shock / hypotension|septic shock                                                           3100
cardiovascular|chest pain / ASHD|acute coronary synd

#### Observation :
- The same major diagnoses appear in both windows, suggesting the clinical composition is fairly stable between 12h and 24h.
- Sepsis and acute respiratory failure are the two most common primary diagnoses, followed by heart failure, DKA, severe sepsis, pneumonia, and septic shock.
- Counts increase at 24h because additional diagnoses are documented, but the overall ranking changes very little.
- These diagnoses will be useful later for labeling and interpreting discovered phenotypes, especially clusters related to sepsis, respiratory failure, or cardiovascular disease.

## Load and inspect `medication.csv`

In [32]:
medication = pd.read_csv(
    data_folder / "medication.csv",
    low_memory=False
)

print("Shape:")
print(medication.shape)

print("\nColumns:")
print(medication.columns.tolist())

print("\nFirst 5 rows:")
medication.head()

Shape:
(7301853, 15)

Columns:
['medicationid', 'patientunitstayid', 'drugorderoffset', 'drugstartoffset', 'drugivadmixture', 'drugordercancelled', 'drugname', 'drughiclseqno', 'dosage', 'routeadmin', 'frequency', 'loadingdose', 'prn', 'drugstopoffset', 'gtc']

First 5 rows:


,medicationid,patientunitstayid,drugorderoffset,drugstartoffset,drugivadmixture,drugordercancelled,drugname,drughiclseqno,dosage,routeadmin,frequency,loadingdose,prn,drugstopoffset,gtc
0,7426715,141168,309,666,No,No,METOPROLOL TARTRATE 25 MG PO TABS,2102.0,25 3,PO,Q12H SCH,NaN,No,1826,0
1,9643232,141168,1847,1832,No,No,3 ML - IPRATROPIUM-ALBUTEROL 0.5-2.5 (3) MG/...,NaN,3 1,NEBULIZATION,Q4H Resp PRN,NaN,Yes,2047,0
2,10270090,141168,296,1386,No,No,ASPIRIN EC 81 MG PO TBEC,1820.0,81 3,PO,Daily,NaN,No,2390,0
3,9496768,141168,2048,2029,No,No,3 ML - IPRATROPIUM-ALBUTEROL 0.5-2.5 (3) MG/...,NaN,3 1,NEBULIZATION,Q4H Resp PRN,NaN,Yes,2390,0
4,11259680,141168,117,246,No,No,ENOXAPARIN SODIUM 40 MG/0.4ML SC SOLN,NaN,40 3,SC,Daily,NaN,No,1721,0


#### Observation :
- This is a large repeated medication-order table with $7.3$ million rows; one ICU stay can have many medication records.
- `drugname`, `dosage`, `routeadmin`, and `frequency` describe the treatment, while `drugstartoffset` and `drugstopoffset` provide timing.
- `drugorderoffset` shows when the order was placed, whereas `drugstartoffset` is more relevant when determining whether a medication started within the 12h or 24h window.
- `drugordercancelled` is important because cancelled orders should not be treated the same as active treatment.
- Medications may be useful for characterizing clusters and treatment patterns, but we should be cautious about using them as core physiological clustering features.

## Check medication coverage and cancellation

In [33]:
print("Unique ICU stays:")
print(medication["patientunitstayid"].nunique())

print("\nMissing drug start offsets:")
print(medication["drugstartoffset"].isnull().sum())

print("\nDrug order cancelled:")
print(medication["drugordercancelled"].value_counts(dropna=False))

medication_valid = medication[
    medication["drugordercancelled"] != "Yes"
]

medication_12h = medication_valid[
    (medication_valid["drugstartoffset"] >= 0) &
    (medication_valid["drugstartoffset"] <= 720)
]

medication_24h = medication_valid[
    (medication_valid["drugstartoffset"] >= 0) &
    (medication_valid["drugstartoffset"] <= 1440)
]

print("\nICU stays with medication started in first 12 hours:")
print(medication_12h["patientunitstayid"].nunique())

print("\nICU stays with medication started in first 24 hours:")
print(medication_24h["patientunitstayid"].nunique())

Unique ICU stays:
165837

Missing drug start offsets:
0

Drug order cancelled:
drugordercancelled
No     7096580
Yes     205273
Name: count, dtype: int64

ICU stays with medication started in first 12 hours:
151316

ICU stays with medication started in first 24 hours:
159754


#### Observation :
- The table covers $165,837$ ICU stays, so medication data are available for a large portion of the cohort.
- `drugstartoffset` has no missing values, which makes medication timing reliable for window filtering.
- Only about $2.8\%$ of medication orders are cancelled, so most records represent valid orders.
- $151,316$ stays have a medication started within 12h, increasing to $159,754$ within 24h.
- Medication data therefore have strong early coverage and should be useful mainly for describing treatment patterns across discovered clusters.

## Check most common medications in 12h and 24h

In [34]:
print("Top 15 medications started in first 12 hours:")
print(
    medication_12h["drugname"]
    .value_counts()
    .head(15)
)

print("\nTop 15 medications started in first 24 hours:")
print(
    medication_24h["drugname"]
    .value_counts()
    .head(15)
)

Top 15 medications started in first 12 hours:
drugname
1000 ML  -  SODIUM CHLORIDE 0.9 % IV SOLN             12587
ACETAMINOPHEN                                         10645
DEXTROSE 50%-WATER                                    10320
ACETAMINOPHEN 325 MG PO TABS                           9893
SODIUM CHLORIDE 0.9%                                   9427
SODIUM CHLORIDE 0.9 % IV : 1000 ML                     9241
TYLENOL                                                8937
1000 ML FLEX CONT : SODIUM CHLORIDE 0.9 % IV SOLN      8878
473 ML  -  CHLORHEXIDINE GLUCONATE 0.12 % MT SOLN      8471
POTASSIUM CHLORIDE CRYS ER 20 MEQ PO TBCR              8470
PANTOPRAZOLE SODIUM 40 MG IV SOLR                      7987
100 ML  -  POTASSIUM CHLORIDE 20 MEQ/100ML IV SOLN     7678
POTASSIUM CHLORIDE 20 MEQ PO PACK                      7229
PEPCID                                                 6571
ZOFRAN                                                 6397
Name: count, dtype: int64

Top 15 medications

#### Observation :
- The most common medications are broadly similar in the 12h and 24h windows, with larger counts at 24h.
- IV saline, acetaminophen, potassium replacement, acid-suppression drugs, and dextrose are among the most frequent treatments.
- The same medication can appear under different names or formulations—for example, acetaminophen vs. Tylenol and several sodium chloride/potassium chloride entries.
- Therefore, if medication data are used later, drug-name standardization/grouping will be necessary before meaningful cluster comparisons.
- Medication patterns are better suited for clinical interpretation of phenotypes than as primary clustering variables.

## Load and Inspect `infusionDrug.csv`

In [35]:
infusion_sample = pd.read_csv(
    data_folder / "infusionDrug.csv",
    nrows=5
)

print("Columns:")
print(infusion_sample.columns.tolist())

print("\nFirst 5 rows:")
infusion_sample

Columns:
['infusiondrugid', 'patientunitstayid', 'infusionoffset', 'drugname', 'drugrate', 'infusionrate', 'drugamount', 'volumeoffluid', 'patientweight']

First 5 rows:


,infusiondrugid,patientunitstayid,infusionoffset,drugname,drugrate,infusionrate,drugamount,volumeoffluid,patientweight
0,1953469,242040,457,Milrinone (mcg/kg/min),0.43,11.8,20.0,100.0,91.7
1,1998443,242082,425,Norepinephrine (mcg/min),10.93,41.0,4.0,250.0,NaN
2,1968206,242082,125,Norepinephrine (mcg/min),7.00,26.3,4.0,250.0,NaN
3,1991487,242082,665,NS (ml/hr),200.00,200.0,NaN,NaN,NaN
4,1969910,242082,55,Norepinephrine (mcg/min),2.13,8.0,4.0,250.0,NaN


#### Observation :
- This is a repeated infusion-medication table, with each record timed using `infusionoffset`.
- It contains clinically important continuous therapies such as norepinephrine and milrinone, as well as IV fluids.
- `drugrate` and `infusionrate` provide treatment intensity, but units vary by medication, so rates cannot be compared directly across different drugs.
- Infusions, especially vasopressors, may be valuable for interpreting high-severity or shock-related clusters.
- Like medications, infusion treatments are better suited mainly for cluster characterization and validation rather than the core physiologic clustering features.

## Check infusion coverage in 12h and 24h

In [36]:
infusion_file = data_folder / "infusionDrug.csv"

total_rows = 0
all_stays = set()
stays_12h = set()
stays_24h = set()

for chunk in pd.read_csv(
    infusion_file,
    usecols=["patientunitstayid", "infusionoffset"],
    chunksize=1_000_000
):
    total_rows += len(chunk)

    all_stays.update(chunk["patientunitstayid"].unique())

    infusion_12h = chunk[
        (chunk["infusionoffset"] >= 0) &
        (chunk["infusionoffset"] <= 720)
    ]

    infusion_24h = chunk[
        (chunk["infusionoffset"] >= 0) &
        (chunk["infusionoffset"] <= 1440)
    ]

    stays_12h.update(infusion_12h["patientunitstayid"].unique())
    stays_24h.update(infusion_24h["patientunitstayid"].unique())

print("Total rows:", total_rows)
print("Unique ICU stays:", len(all_stays))

print("\nICU stays with infusion data in first 12 hours:")
print(len(stays_12h))

print("\nICU stays with infusion data in first 24 hours:")
print(len(stays_24h))

Total rows: 4803719
Unique ICU stays: 73547

ICU stays with infusion data in first 12 hours:
62957

ICU stays with infusion data in first 24 hours:
67810


#### Observation :
- The table contains $4.8$ million infusion records but covers only $73,547$ ICU stays, much less than the vital/lab tables.
- $62,957$ stays have infusion data within 12h, increasing to $67,810$ within 24h.
- Because infusion data exist for only a subset of patients, requiring them as clustering features would substantially reduce the cohort.
- Infusion information is therefore better used for post-clustering interpretation, such as identifying clusters with greater vasopressor or fluid support.
  

## Check common infusions in 12h and 24h

In [37]:
infusion_counts_12h = {}
infusion_counts_24h = {}

for chunk in pd.read_csv(
    infusion_file,
    usecols=["infusionoffset", "drugname"],
    chunksize=1_000_000
):
    data_12h = chunk[
        (chunk["infusionoffset"] >= 0) &
        (chunk["infusionoffset"] <= 720)
    ]

    data_24h = chunk[
        (chunk["infusionoffset"] >= 0) &
        (chunk["infusionoffset"] <= 1440)
    ]

    counts_12h = data_12h["drugname"].value_counts()
    counts_24h = data_24h["drugname"].value_counts()

    for drug, count in counts_12h.items():
        infusion_counts_12h[drug] = infusion_counts_12h.get(drug, 0) + count

    for drug, count in counts_24h.items():
        infusion_counts_24h[drug] = infusion_counts_24h.get(drug, 0) + count

print("Top 15 infusions in first 12 hours:")
print(pd.Series(infusion_counts_12h).sort_values(ascending=False).head(15))

print("\nTop 15 infusions in first 24 hours:")
print(pd.Series(infusion_counts_24h).sort_values(ascending=False).head(15))

Top 15 infusions in first 12 hours:
Norepinephrine (ml/hr)      43818
Propofol (ml/hr)            42767
Norepinephrine (mcg/min)    37386
Insulin (ml/hr)             36118
Insulin (units/hr)          35339
Propofol (mcg/kg/min)       34981
Fentanyl (mcg/hr)           27727
Fentanyl (ml/hr)            21007
Heparin (ml/hr)             20012
NS (ml/hr)                  17665
Midazolam (mg/hr)           15408
NSS (ml/hr)                 14560
Amiodarone (ml/hr)          14536
Heparin (units/hr)          14073
Nicardipine (ml/hr)         12706
dtype: int64

Top 15 infusions in first 24 hours:
Norepinephrine (ml/hr)      84943
Propofol (ml/hr)            77125
Norepinephrine (mcg/min)    65483
Insulin (units/hr)          65313
Insulin (ml/hr)             63392
Propofol (mcg/kg/min)       58558
Fentanyl (mcg/hr)           51687
Fentanyl (ml/hr)            45969
Heparin (ml/hr)             37743
NS (ml/hr)                  30208
Midazolam (mg/hr)           28527
Amiodarone (ml/hr)          28

#### Observation :
- Norepinephrine is the most frequently recorded infusion, making vasopressor use potentially useful for identifying shock/high-severity phenotypes.
- Propofol, fentanyl, and midazolam are also common, reflecting sedation/ventilation-related treatment patterns.
- Insulin, heparin, IV fluids, and cardiac drugs are frequently recorded as well.
- The same drug appears under different units/formats (ml/hr, mcg/min, units/hr), so infusion names and units would need standardization before detailed analysis.
- These data are best used to describe treatment differences between clusters, rather than as required clustering inputs because infusion coverage is limited.